# 05b Neural rankers: feature MLP and history attention

**Purpose.** The neural part of PROJECT_PLAN section 5 (item 4c), on a Colab GPU. Both models are compared with the final LambdaMART on the same validation searches:

1. **Feature MLP (E27, this part):** the same 19 inputs as the final LambdaMART (query length, relevance, lagged popularity, pooled history, item-item CF) with a ranking loss. The control: how much of LambdaMART's result is the features and how much the trees?
2. **History attention (next part):** the MLP plus attention over the shopper's last history events, weighted by the query and the candidate.

**Rules (as for LambdaMART, notebook 05).** Training on all clean training searches minus the same inner early-stopping set (3,757 dev searches); clamped searches and searches without an engaged candidate excluded; candidates were shuffled when the features were built; evaluation with our exact tie-aware evaluator on all validation lists, `validation_clean` as the main population; paired per-search differences. One fixed configuration per model, one debugging cycle, no tuning (PROJECT_PLAN 5, 28 Oct).

**Score with a popularity residual.** score = a x log(1 + lagged popularity) + f(inputs). The last layer of f starts at zero and a at 1, so the untrained model ranks exactly by lagged popularity, the deployable reference; training only has to learn what improves on it.

**Loss.** Listwise softmax cross-entropy over the complete list: the model's softmax over the candidates is pushed towards a target distribution proportional to the labels (0-3), so purchases count three times a click, as in our NDCG gains.

**How to run on Colab.** Copy to `MyDrive/BT4222 Group Project/processed/` (in addition to the v3 files from E21): `cf/cf1_train_dev.parquet`, `cf/cf1_train_rest.parquet`, `cf/cf1_validation.parquet` (05 section 10) and `neural/lambdamart_validation_scores.parquet` (05 section 14). Open this notebook from GitHub (branch `BRANCH` below), choose a GPU runtime and *Run all*. Results go to `processed/neural/`; download the executed notebook afterwards. Without a GPU the notebook runs a **smoke test** (2,000 training and 1,000 validation searches, one epoch) whose numbers are not results.

In [1]:
# torch first: on Windows, importing numpy / pandas before torch can break torch's DLL loading.
import torch
import torch.nn as nn

from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BRANCH = "feat/models"  # branch to clone on Colab; "main" once merged
REPO_ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/evaluation.py").is_file()), None)
if REPO_ROOT is None and IN_COLAB:  # opened from GitHub: clone the (public) repository, or update an earlier clone
    import subprocess
    REPO_ROOT = Path("/content/BT4222-group17-jdsearch")
    if (REPO_ROOT / ".git").is_dir():  # a second run in the same runtime: bring the clone to the branch's latest commit
        subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "reset", "--hard", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                        "https://github.com/liamkunert/BT4222-group17-jdsearch.git", str(REPO_ROOT)], check=True)
if REPO_ROOT is None:
    raise FileNotFoundError("Open/clone the repository checkout first.")
if IN_COLAB:
    drive.mount("/content/drive")
    PROCESSED = Path("/content/drive/MyDrive/BT4222 Group Project/processed")
    DATA_DIR = Path("/content/drive/MyDrive/BT4222 Group Project/data")
else:
    PROCESSED = REPO_ROOT / "data/processed"
    DATA_DIR = REPO_ROOT / "data"
BEHAVIOUR, CATALOGUE = DATA_DIR / "user_behavior_data.txt", DATA_DIR / "product_meta_data.txt"
NEURAL_DIR = PROCESSED / "neural"
NEURAL_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT))
from scripts.evaluation import evaluate, paired_difference, summarise
from scripts.jd_data import SECONDS_PER_DAY, iter_records
from scripts.query_diagnostics import tokenize

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE = DEVICE.type != "cuda"  # smoke test without a GPU: small subsets, one epoch
TAG = "_smoke" if SMOKE else ""
RETRAIN = False  # True: train a neural model again even if its results are already in processed/neural/
SEED, MODEL_SEED = 4222, 1
pd.options.display.float_format = "{:,.4f}".format
meta = json.loads((PROCESSED / "features_v3_meta.json").read_text())
searches = pd.read_parquet(PROCESSED / "search_table.parquet")
assignment = pd.read_csv(PROCESSED / "split_v1.csv", index_col="session_id")
print(f"device {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'}); smoke test: {SMOKE}; "
      f"torch {torch.__version__}; data {PROCESSED}")

Mounted at /content/drive
device cuda (Tesla T4); smoke test: False; torch 2.11.0+cu130; data /content/drive/MyDrive/BT4222 Group Project/processed


## 1. Populations and inputs

The same training and early-stopping searches as LambdaMART in notebook 05 (reproduced here and checked by their counts), and the same 19 inputs as the final LambdaMART.

In [2]:
dev_ids = assignment.index[(assignment.split == "train") & assignment.dev_subsample]
dev_info = searches.loc[dev_ids]
dev_pool = dev_ids[~(dev_info.final_gap_clamped | (dev_info.n_engaged == 0) | (dev_info.n_candidates == 1)).to_numpy()]
inner_ids = np.sort(np.random.default_rng(SEED).choice(dev_pool.to_numpy(), size=round(0.1 * len(dev_pool)), replace=False))
train_info = searches.loc[assignment.index[assignment.split == "train"]]
full_pool = train_info.index[(~train_info.final_gap_clamped & (train_info.n_engaged > 0)
                              & (train_info.n_candidates > 1)).to_numpy()]
fit_ids = np.setdiff1d(full_pool.to_numpy(), inner_ids)
assert (len(fit_ids), len(inner_ids)) == (121_989, 3_757)  # as in 05, section 11
validation_ids = assignment.index[assignment.split == "validation"].to_numpy()
if SMOKE:
    rng = np.random.default_rng(SEED)
    fit_ids = np.sort(rng.choice(fit_ids, 2_000, replace=False))
    inner_ids = np.sort(rng.choice(inner_ids, 500, replace=False))
    validation_ids = np.sort(rng.choice(validation_ids, 1_000, replace=False))
validation_sizes = searches.loc[validation_ids, "n_candidates"]

G = meta["groups"]
CF_COLUMNS = ["cf_sum", "cf_max", "cf_mean", "cf_count", "cf_history_in_graph"]
INPUTS = G["G0_search"] + G["G1_relevance"] + G["G2a_popularity_deployable"] + G["H1_pooled"] + CF_COLUMNS
POPULARITY = "pop_history_lag1d"
assert len(INPUTS) == 19


def load(split, sessions):
    """Session ids, wids, raw inputs (float32) and labels for the given searches of one split."""
    path = PROCESSED / f"features_v3_{split}.parquet"
    cf_path = PROCESSED / "cf" / f"cf1_{split}.parquet"
    ids = pq.read_table(path, columns=["session_id", "wid"]).to_pandas()
    cf_ids = pq.read_table(cf_path, columns=["session_id", "wid"]).to_pandas()
    assert ids.equals(cf_ids), f"CF rows of {split} do not line up with the v3 rows"
    keep = ids.session_id.isin(sessions).to_numpy()
    X = np.empty((int(keep.sum()), len(INPUTS)), dtype=np.float32)
    for j, column in enumerate(INPUTS):
        source = cf_path if column in CF_COLUMNS else path
        X[:, j] = pq.read_table(source, columns=[column]).column(0).to_numpy()[keep]
    labels = pq.read_table(path, columns=["label"]).column(0).to_numpy()[keep]
    return ids.session_id.to_numpy()[keep], ids.wid.to_numpy()[keep], X, labels


start = time.time()
parts = [load(split, fit_ids) for split in ("train_dev", "train_rest")]
fit = [np.concatenate([p[k] for p in parts]) for k in range(4)]
inner = load("train_dev", inner_ids)
valid = load("validation", validation_ids)
del parts
for name, (sid, _, X, y) in {"fit": fit, "inner": inner, "validation": valid}.items():
    print(f"{name}: {len(np.unique(sid)):,} searches, {len(sid):,} rows, engaged share {np.mean(y >= 1):.4f}")
print(f"loaded in {time.time() - start:.0f} s")

fit: 121,989 searches, 10,998,466 rows, engaged share 0.0211
inner: 3,757 searches, 341,134 rows, engaged share 0.0207
validation: 20,000 searches, 1,781,573 rows, engaged share 0.0209
loaded in 45 s


## 2. Input transformation

Fitted on the training rows only: inputs that are non-negative with a maximum above 10 (counts such as popularity, history matches, BM25) are log-transformed with log(1 + x); all inputs are standardised; missing values (unknown metadata, candidates outside the CF graph) become 0 after standardisation, with a 0/1 missing flag for every input that has missing values. Lagged popularity also enters the residual term as log(1 + x).

In [3]:
class Inputs:
    """Log-transform, standardise and flag missing values; statistics from the training rows only."""

    def __init__(self, X):
        self.log = (np.nanmin(X, axis=0) >= 0) & (np.nanmax(X, axis=0) > 10)
        self.flag = np.isnan(X).any(axis=0)
        Z = self._log(X)
        self.mean, self.std = np.nanmean(Z, axis=0), np.nanstd(Z, axis=0) + 1e-6

    def _log(self, X):
        Z = X.copy()
        Z[:, self.log] = np.log1p(Z[:, self.log])
        return Z

    def __call__(self, X):
        Z = (self._log(X) - self.mean) / self.std
        missing = np.isnan(X[:, self.flag]).astype(np.float32)
        return np.hstack([np.nan_to_num(Z, nan=0.0), missing]).astype(np.float32)


transform = Inputs(fit[2])
pop_j = INPUTS.index(POPULARITY)
print(f"log-transformed: {[c for c, l in zip(INPUTS, transform.log) if l]}")
print(f"missing flags: {[c for c, f in zip(INPUTS, transform.flag) if f]}")


class Lists:
    """One split on the device: transformed inputs, residual term, labels and the row range of every search."""

    def __init__(self, data):
        sid, wid, X, y = data
        bounds = np.r_[0, np.flatnonzero(np.diff(sid)) + 1, len(sid)]
        assert len(bounds) - 1 == len(np.unique(sid)), "rows of a search must be contiguous"
        self.sid, self.wid, self.labels_np = sid, wid, y
        self.start, self.length = bounds[:-1], np.diff(bounds)
        self.x = torch.from_numpy(transform(X)).to(DEVICE)
        self.log_pop = torch.from_numpy(np.log1p(np.nan_to_num(X[:, pop_j], nan=0.0))).to(DEVICE)
        self.y = torch.from_numpy(y.astype(np.float32)).to(DEVICE)

    def __len__(self):
        return len(self.start)

    def batch(self, searches):
        """Padded row index (searches x longest list), its mask and the search positions as a tensor."""
        start = torch.as_tensor(self.start[searches], device=DEVICE)
        length = torch.as_tensor(self.length[searches], device=DEVICE)
        offsets = torch.arange(int(length.max()), device=DEVICE)
        mask = offsets[None, :] < length[:, None]
        index = torch.where(mask, start[:, None] + offsets[None, :], torch.zeros_like(mask, dtype=torch.long))
        return index, mask, torch.as_tensor(searches, device=DEVICE)


data = {"fit": Lists(fit), "inner": Lists(inner), "validation": Lists(valid)}
del fit, inner
print({name: len(lists) for name, lists in data.items()},
      f"inputs after transformation: {data['fit'].x.shape[1]}")

log-transformed: ['query_len', 'title_shared', 'bm25', 'title_len', 'pop_history_lag1d', 'pooled_product', 'pooled_shop_other', 'pooled_brand_other', 'pooled_L3', 'cf_sum', 'cf_count', 'cf_history_in_graph']
missing flags: ['title_shared', 'title_coverage', 'title_jaccard', 'bm25', 'title_len', 'query_in_brand', 'query_in_cate3', 'pooled_shop_other', 'pooled_brand_other', 'pooled_L3', 'cf_sum', 'cf_max', 'cf_mean', 'cf_count']
{'fit': 121989, 'inner': 3757, 'validation': 20000} inputs after transformation: 33


## 3. Model, loss and training loop

Shared by both neural models: a list is scored as a whole (padded to its longest member), the loss is the listwise softmax cross-entropy, and early stopping uses NDCG@10 with linear gains on the inner set (the final numbers come from our evaluator).

In [4]:
class FeatureMLP(nn.Module):
    """score = a * log(1 + lagged popularity) + MLP(inputs); last layer at zero, a = 1: starts as popularity."""

    def __init__(self, n_inputs, hidden=(128, 64), dropout=0.1):
        super().__init__()
        layers, width = [], n_inputs
        for h in hidden:
            layers += [nn.Linear(width, h), nn.ReLU(), nn.Dropout(dropout)]
            width = h
        self.body = nn.Sequential(*layers)
        self.out = nn.Linear(width, 1)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)
        self.alpha = nn.Parameter(torch.tensor(1.0))

    def forward(self, lists, index, mask, searches):
        x, log_pop = lists.x[index], lists.log_pop[index]
        return self.alpha * log_pop + self.out(self.body(x)).squeeze(-1)


def listwise_loss(scores, labels, mask):
    """Softmax cross-entropy against targets proportional to the labels (lists with an engaged candidate only)."""
    scores = scores.masked_fill(~mask, -1e9)
    target = labels * mask
    target = target / target.sum(dim=1, keepdim=True).clamp(min=1e-9)
    return -(target * torch.log_softmax(scores, dim=1)).sum(dim=1).mean()


DISCOUNT = 1 / torch.log2(torch.arange(2, 12, device=DEVICE, dtype=torch.float32))


def batch_ndcg(scores, labels, mask, k=10):
    """NDCG@k with linear gains per list (ties in sort order; for early stopping only)."""
    scores = scores.masked_fill(~mask, -float("inf"))
    labels = labels * mask
    top = torch.topk(scores, min(k, scores.shape[1]), dim=1).indices
    dcg = (labels.gather(1, top) * DISCOUNT[:top.shape[1]]).sum(dim=1)
    ideal = (torch.topk(labels, top.shape[1], dim=1).values * DISCOUNT[:top.shape[1]]).sum(dim=1)
    keep = ideal > 0
    return (dcg[keep] / ideal[keep]).sum().item(), int(keep.sum())


@torch.no_grad()
def predict(model, lists, batch_size=512):
    """Scores for every row of a split, in row order."""
    model.eval()
    scores = torch.empty(len(lists.y), device=DEVICE)
    for lo in range(0, len(lists), batch_size):
        index, mask, searches = lists.batch(np.arange(lo, min(lo + batch_size, len(lists))))
        scores[index[mask]] = model(lists, index, mask, searches)[mask]
    return scores


@torch.no_grad()
def inner_ndcg(model, batch_size=512):
    model.eval()
    lists, total, n = data["inner"], 0.0, 0
    for lo in range(0, len(lists), batch_size):
        index, mask, searches = lists.batch(np.arange(lo, min(lo + batch_size, len(lists))))
        s, m = batch_ndcg(model(lists, index, mask, searches), lists.y[index], mask)
        total, n = total + s, n + m
    return total / n


def train(model, epochs=10, patience=2, batch_size=256, lr=1e-3, label="", embedding_decay=0.0):
    """Adam on the fit searches, early stopping on the inner set's NDCG@10; returns the best model and a log.

    embedding_decay > 0 (E31): AdamW with decoupled weight decay on the embedding tables only."""
    torch.manual_seed(MODEL_SEED)
    rng = np.random.default_rng(MODEL_SEED)
    model.to(DEVICE)
    if embedding_decay:
        tables = [m.weight for m in model.modules() if isinstance(m, nn.Embedding)]
        rest = [p for p in model.parameters() if all(p is not t for t in tables)]
        optimiser = torch.optim.AdamW([{"params": tables, "weight_decay": embedding_decay},
                                       {"params": rest, "weight_decay": 0.0}], lr=lr)
    else:
        optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    history = [{"epoch": 0, "loss": np.nan, "inner_ndcg_at_10": inner_ndcg(model), "seconds": 0.0}]
    best, best_state, waited = history[0]["inner_ndcg_at_10"], {k: v.clone() for k, v in model.state_dict().items()}, 0
    lists = data["fit"]
    for epoch in range(1, (1 if SMOKE else epochs) + 1):
        model.train()
        start, losses = time.time(), []
        for batch in np.array_split(rng.permutation(len(lists)), max(1, len(lists) // batch_size)):
            index, mask, searches = lists.batch(batch)
            loss = listwise_loss(model(lists, index, mask, searches), lists.y[index], mask)
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            losses.append(loss.item())
        score = inner_ndcg(model)
        history.append({"epoch": epoch, "loss": float(np.mean(losses)), "inner_ndcg_at_10": score,
                        "seconds": round(time.time() - start, 1)})
        print(f"{label} epoch {epoch}: loss {np.mean(losses):.4f}, inner NDCG@10 {score:.4f}, {time.time() - start:.0f} s")
        if score > best:
            best, best_state, waited = score, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            waited += 1
            if waited >= patience:
                break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history).set_index("epoch")

Check before training: the untrained model must rank exactly by lagged popularity (same scores up to the constant factor a = 1).

In [5]:
torch.manual_seed(MODEL_SEED)
untrained = FeatureMLP(data["fit"].x.shape[1]).to(DEVICE)
check = predict(untrained, data["validation"]).cpu().numpy()
assert np.allclose(check, np.log1p(np.nan_to_num(valid[2][:, pop_j], nan=0.0)), atol=1e-5)
print("untrained model = lagged popularity: ok")

untrained model = lagged popularity: ok


## 4. E27: feature MLP

In [6]:
def saved_run(name):
    """Scores, training log and summary of an earlier run in processed/neural/, if present (and RETRAIN is off)."""
    files = [NEURAL_DIR / f"{name}_{kind}{TAG}.{ext}" for kind, ext in
             [("validation_scores", "parquet"), ("history", "csv"), ("summary", "json")]]
    if RETRAIN or not (files[0].is_file() and files[2].is_file()):  # the training log is optional (first MLP run)
        return None
    scores = pd.read_parquet(files[0])
    assert (scores.session_id.to_numpy() == valid[0]).all() and (scores.wid.to_numpy() == valid[1]).all()
    print(f"{name}: loaded from {files[0]}")
    history = pd.read_csv(files[1], index_col="epoch") if files[1].is_file() else None
    return scores.score.to_numpy(), history, json.loads(files[2].read_text())


def save_run(name, scores, history, summary):
    pd.DataFrame({"session_id": valid[0], "wid": valid[1], "score": scores}).to_parquet(
        NEURAL_DIR / f"{name}_validation_scores{TAG}.parquet", index=False)
    history.to_csv(NEURAL_DIR / f"{name}_history{TAG}.csv")
    (NEURAL_DIR / f"{name}_summary{TAG}.json").write_text(json.dumps(summary, indent=2))


saved = saved_run("mlp")
if saved is None:
    torch.manual_seed(MODEL_SEED)
    start = time.time()
    mlp, mlp_history = train(FeatureMLP(data["fit"].x.shape[1]), label="MLP")
    mlp_scores = predict(mlp, data["validation"]).cpu().numpy()
    mlp_summary = {"model": "feature MLP", "inputs": INPUTS, "transformed_inputs": int(data["fit"].x.shape[1]),
                   "fit_searches": len(data["fit"]), "inner_searches": len(data["inner"]), "smoke": SMOKE,
                   "device": str(DEVICE), "minutes": round((time.time() - start) / 60, 1),
                   "alpha": float(mlp.alpha.item()), "best_epoch": int(mlp_history.inner_ndcg_at_10.idxmax())}
    save_run("mlp", mlp_scores, mlp_history, mlp_summary)
else:
    mlp_scores, mlp_history, mlp_summary = saved
display(pd.Series({k: v for k, v in mlp_summary.items() if k != "inputs"}).to_frame("feature MLP"))
mlp_history

mlp: loaded from /content/drive/MyDrive/BT4222 Group Project/processed/neural/mlp_validation_scores.parquet


,feature MLP
model,feature MLP
transformed_inputs,33
fit_searches,121989
inner_searches,3757
smoke,False
device,cuda
minutes,0.3000
alpha,0.8318
best_epoch,4


,loss,inner_ndcg_at_10,seconds
epoch,,,
0,NaN,0.3366,0.0000
1,3.5780,0.3822,2.2000
2,3.5418,0.3839,1.9000
3,3.5337,0.3852,1.9000
4,3.5296,0.3869,1.9000
5,3.5268,0.3859,1.9000
6,3.5247,0.3846,1.9000


## 5. Evaluation against the references

All validation lists with our evaluator; `validation_clean` is the main population. References on the same searches: lagged popularity, and the LambdaMART base, M1 and M1 + CF trained on all training searches (scores exported from notebook 05, section 14).

In [7]:
sid, wid, X_valid, y_valid = valid
frame = pd.DataFrame({"session_id": sid, "wid": wid, "label": y_valid})
reference_file = NEURAL_DIR / "lambdamart_validation_scores.parquet"
reference = pd.read_parquet(reference_file) if reference_file.is_file() else None
rankings = {"feature MLP": mlp_scores, "lagged history popularity": np.nan_to_num(X_valid[:, pop_j], nan=0.0)}
if reference is not None:  # rows in v3 order, like `valid`
    reference = reference[reference.session_id.isin(validation_ids).to_numpy()]
    assert (reference.session_id.to_numpy() == sid).all() and (reference.wid.to_numpy() == wid).all()
    rankings.update({"LambdaMART base": reference["base"].to_numpy(), "LambdaMART M1": reference["m1"].to_numpy(),
                     "LambdaMART M1 + CF": reference["m1_cf"].to_numpy()})
per_search = {name: evaluate(frame.assign(score=s), "score", expected_sizes=validation_sizes)
              for name, s in rankings.items()}
all_ids = per_search["feature MLP"].index
clean_ids = all_ids[~searches.final_gap_clamped.reindex(all_ids).to_numpy()]
rows = {}
for name in rankings:
    row = summarise(per_search[name].loc[clean_ids]).iloc[0].to_dict()
    row["ndcg_all_searches"] = per_search[name].ndcg.mean()
    if name != "feature MLP":
        d = paired_difference(per_search["feature MLP"].loc[clean_ids], per_search[name].loc[clean_ids])
        row.update(mlp_minus_this=d[0], low=d[1], high=d[2])
    rows[name] = row
results = pd.DataFrame(rows).T
results["searches"] = results.searches.astype(int)
results

,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,ndcg_all_searches,mlp_minus_this,low,high
feature MLP,18844,0.3889,0.3837,0.3940,0.5732,0.3942,0.4020,NaN,NaN,NaN
lagged history popularity,18844,0.3374,0.3327,0.3423,0.5212,0.3352,0.3369,0.0515,0.0483,0.0545
LambdaMART base,18844,0.3500,0.3453,0.3547,0.5361,0.3480,0.3490,0.0390,0.0361,0.0418
LambdaMART M1,18844,0.3887,0.3837,0.3941,0.5705,0.3944,0.4029,0.0002,-0.0012,0.0019
LambdaMART M1 + CF,18844,0.3918,0.3866,0.3973,0.5733,0.3978,0.4060,-0.0028,-0.0044,-0.0012


**Reading the result (E27, Colab T4).**
- The feature MLP reaches **0.389** on `validation_clean`, the same as LambdaMART M1 (+0.0002 [-0.0012, 0.0019]) and **-0.0028 [-0.0044, -0.0012]** below the final LambdaMART M1 + CF (0.392); +0.039 over the LambdaMART base, +0.052 over lagged popularity. Within the expectation (0.37-0.39, at most LambdaMART).
- Best epoch 4 of 6 (2 seconds per epoch on a T4); the popularity weight a moved from 1 to 0.83.
- Reading: with the same inputs, a neural ranker does as well as the trees on the base and pooled history, but does not extract the small CF gain. The result is carried by the inputs, not by the model family.

## 6. E28: history attention over the last 50 events

The feature MLP sees history only as four pooled match counts and five CF similarities. The attention model also sees the **individual events**: for every candidate it weights the shopper's last 50 history actions by how relevant each one is to this candidate and this query, and adds the weighted summary to the MLP's inputs.

- **Item representation:** the sum of learned 32-dimensional embeddings of the product, its shop, brand and L3 category path. Vocabularies come from the fit searches only (candidates and histories, at least 5 occurrences); rarer and missing values share the zero vector, so an unseen product is still represented by its shop, brand and category.
- **Event:** item representation + embeddings of the action type and the age bucket (under 10 minutes, 10 minutes to 1 hour, 1 hour to 1 day, 1-7, 7-30, 30-90, 90-180, over 180 days).
- **Query:** mean of its term embeddings (first 8 terms, vocabulary from the fit queries).
- **Attention:** weight of event t for candidate c = softmax over t of (W c + V q) . e_t / sqrt(32); summary u = sum of weights x events. Inputs of the scoring MLP: the 33 transformed features, c, u and c x u.
- Same residual on lagged popularity (zero-initialised last layer), loss, optimiser, early stopping, populations and evaluation as the feature MLP; one configuration, no tuning.

In [8]:
HIST_LEN, EMB_DIM, MIN_COUNT, QUERY_LEN = 50, 32, 5, 8
TYPE_ID = {"CLICK": 1, "CART": 2, "ORD": 3, "FLW": 4}
AGE_EDGES = np.array([10 / 1440, 1 / 24, 1, 7, 30, 90, 180])  # days; buckets 1-8 (0 = padding)
needed = set(np.unique(np.concatenate([lists.sid for lists in data.values()])).tolist())
start = time.time()
raw = {}
for sid, fields in iter_records(BEHAVIOUR):
    if sid not in needed:
        continue
    gaps = np.array(fields[6].split("_"), dtype=np.int64)
    age = np.cumsum(gaps[::-1])[::-1][1:] / SECONDS_PER_DAY  # days before the search, as in 03
    past = fields[4].split("_") if fields[4] else []
    types = fields[5].split("_") if fields[5] else []
    raw[sid] = (np.array([int(t) for t in tokenize(fields[0])[:QUERY_LEN]], dtype=np.int64),
                np.array(past[-HIST_LEN:], dtype=np.int64),
                np.array([TYPE_ID[t] for t in types[-HIST_LEN:]], dtype=np.int64),
                np.searchsorted(AGE_EDGES, age[-HIST_LEN:], side="right") + 1)
print(f"histories of {len(raw):,} searches read in {time.time() - start:.0f} s")

# product attributes from the catalogue (brand, shop, L3 path), only for products that occur here
start = time.time()
products = np.unique(np.concatenate([lists.wid for lists in data.values()] + [r[1] for r in raw.values()]))
catalogue = pd.read_csv(CATALOGUE, sep="\t", quoting=3, dtype="float64", na_values=["-1", ""],
                        usecols=["wid", "brand_id", "cate_id_1", "cate_id_2", "cate_id_3", "shop_id"])
catalogue = catalogue[catalogue.wid.isin(products)]
catalogue["wid"] = catalogue.wid.astype(np.int64)
catalogue = catalogue.drop_duplicates("wid").sort_values("wid")
path = catalogue[["cate_id_1", "cate_id_2", "cate_id_3"]]
l3 = pd.util.hash_pandas_object(path.fillna(-1), index=False).to_numpy().view(np.int64)
ATTRIBUTES = {"product": None,
              "shop": catalogue.shop_id.fillna(-1).to_numpy(np.int64),
              "brand": catalogue.brand_id.fillna(-1).to_numpy(np.int64),
              "L3": np.where(path.isna().any(axis=1).to_numpy(), -1, l3)}
CATALOGUE_WID = catalogue.wid.to_numpy()
del catalogue, path, l3
print(f"{len(products):,} products, {len(CATALOGUE_WID) / len(products):.1%} with a catalogue row; "
      f"{time.time() - start:.0f} s")


def attributes(wids):
    """Product id and its shop, brand and L3 path codes (-1: missing) for an array of wids."""
    pos = np.searchsorted(CATALOGUE_WID, wids)
    found = (pos < len(CATALOGUE_WID)) & (CATALOGUE_WID[np.minimum(pos, len(CATALOGUE_WID) - 1)] == wids)
    out = {"product": wids}
    for name in ("shop", "brand", "L3"):
        values = ATTRIBUTES[name][np.minimum(pos, len(CATALOGUE_WID) - 1)]
        out[name] = np.where(found, values, -1)
    return out


def vocabulary(values):
    """Sorted values seen at least MIN_COUNT times (missing = -1 excluded)."""
    kept, counts = np.unique(values[values != -1], return_counts=True)
    return kept[counts >= MIN_COUNT]


def encode(values, kept):
    """1-based index into the vocabulary; 0 for rare, unseen or missing values."""
    pos = np.searchsorted(kept, values)
    ok = (values != -1) & (pos < len(kept)) & (kept[np.minimum(pos, len(kept) - 1)] == values)
    return np.where(ok, pos + 1, 0).astype(np.int32)


fit_sids = data["fit"].sid[data["fit"].start]
fit_cand = attributes(data["fit"].wid)
fit_hist = attributes(np.concatenate([raw[s][1] for s in fit_sids]))
VOCAB = {name: vocabulary(np.concatenate([fit_cand[name], fit_hist[name]])) for name in ATTRIBUTES}
VOCAB["term"] = vocabulary(np.concatenate([raw[s][0] for s in fit_sids]))
del fit_cand, fit_hist
print({name: f"{len(v):,}" for name, v in VOCAB.items()})

histories of 145,746 searches read in 20 s
8,363,861 products, 98.1% with a catalogue row; 92 s
{'product': '530,542', 'shop': '183,993', 'brand': '108,269', 'L3': '4,945', 'term': '8,188'}


In [9]:
FIELDS = ["product", "shop", "brand", "L3"]


def attach(lists):
    """Candidate ids (rows x 4), last-50 history (searches x 50 x 6) and query terms (searches x 8) on the device."""
    cand = attributes(lists.wid)
    lists.cand = torch.from_numpy(np.stack([encode(cand[f], VOCAB[f]) for f in FIELDS], axis=1)).to(DEVICE)
    sids = lists.sid[lists.start]
    hist = np.zeros((len(sids), HIST_LEN, len(FIELDS) + 2), dtype=np.int32)
    query = np.zeros((len(sids), QUERY_LEN), dtype=np.int32)
    for i, s in enumerate(sids):
        terms, wids, types, ages = raw[s]
        query[i, :len(terms)] = encode(terms, VOCAB["term"])
        if len(wids):
            a = attributes(wids)
            for j, f in enumerate(FIELDS):
                hist[i, :len(wids), j] = encode(a[f], VOCAB[f])
            hist[i, :len(wids), 4], hist[i, :len(wids), 5] = types, ages
    lists.hist, lists.query = torch.from_numpy(hist).to(DEVICE), torch.from_numpy(query).to(DEVICE)


start = time.time()
for lists in data.values():
    attach(lists)
del raw
print(f"attached in {time.time() - start:.0f} s; candidates with a known product embedding: "
      f"{(data['validation'].cand[:, 0] > 0).float().mean().item():.1%} (validation)")


class HistoryAttention(FeatureMLP):
    """Feature MLP plus attention over the last 50 events, conditioned on the candidate and the query."""

    def __init__(self, n_inputs, dim=EMB_DIM, hidden=(128, 64), dropout=0.1):
        super().__init__(n_inputs + 3 * dim, hidden, dropout)
        self.items = nn.ModuleList([nn.Embedding(len(VOCAB[f]) + 1, dim, padding_idx=0) for f in FIELDS])
        self.action_type = nn.Embedding(len(TYPE_ID) + 1, dim, padding_idx=0)  # not self.type: Module.type exists
        self.age_bucket = nn.Embedding(len(AGE_EDGES) + 2, dim, padding_idx=0)
        self.term = nn.Embedding(len(VOCAB["term"]) + 1, dim, padding_idx=0)
        self.candidate_key, self.query_key = nn.Linear(dim, dim), nn.Linear(dim, dim)
        self.dim = dim

    def item(self, ids):
        return sum(embedding(ids[..., j].long()) for j, embedding in enumerate(self.items))

    def forward(self, lists, index, mask, searches):
        c = self.item(lists.cand[index])                                      # searches x list x dim
        hist = lists.hist[searches]                                            # searches x 50 x 6
        events = self.item(hist[..., :4]) + self.action_type(hist[..., 4].long()) + self.age_bucket(hist[..., 5].long())
        has_event = hist[..., 4] > 0
        terms = lists.query[searches].long()
        q = self.term(terms).sum(dim=1) / (terms > 0).sum(dim=1, keepdim=True).clamp(min=1)
        key = self.candidate_key(c) + self.query_key(q)[:, None, :]
        logits = torch.einsum("bld,btd->blt", key, events) / self.dim ** 0.5
        logits = logits.masked_fill(~has_event[:, None, :], -1e9)
        weights = torch.softmax(logits, dim=-1) * has_event[:, None, :]       # shoppers without history: u = 0
        u = torch.einsum("blt,btd->bld", weights, events)
        x = torch.cat([lists.x[index], c, u, c * u], dim=-1)
        return self.alpha * lists.log_pop[index] + self.out(self.body(x)).squeeze(-1)


torch.manual_seed(MODEL_SEED)
check = predict(HistoryAttention(data["fit"].x.shape[1]).to(DEVICE), data["validation"]).cpu().numpy()
assert np.allclose(check, np.log1p(np.nan_to_num(valid[2][:, pop_j], nan=0.0)), atol=1e-5)
print("untrained attention model = lagged popularity: ok")

attached in 38 s; candidates with a known product embedding: 28.9% (validation)
untrained attention model = lagged popularity: ok


In [10]:
saved = saved_run("attention")
if saved is None:
    torch.manual_seed(MODEL_SEED)
    start = time.time()
    attention, attention_history = train(HistoryAttention(data["fit"].x.shape[1]), label="attention")
    attention_scores = predict(attention, data["validation"]).cpu().numpy()
    attention_summary = {"model": "history attention", "inputs": INPUTS, "history_events": HIST_LEN,
                         "embedding_dim": EMB_DIM, "vocabularies": {k: int(len(v)) for k, v in VOCAB.items()},
                         "parameters": int(sum(p.numel() for p in attention.parameters())),
                         "fit_searches": len(data["fit"]), "smoke": SMOKE, "device": str(DEVICE),
                         "minutes": round((time.time() - start) / 60, 1), "alpha": float(attention.alpha.item()),
                         "best_epoch": int(attention_history.inner_ndcg_at_10.idxmax())}
    save_run("attention", attention_scores, attention_history, attention_summary)
else:
    attention_scores, attention_history, attention_summary = saved
display(pd.Series({k: v for k, v in attention_summary.items() if k not in ("inputs", "vocabularies")})
        .to_frame("history attention"))
attention_history

attention: loaded from /content/drive/MyDrive/BT4222 Group Project/processed/neural/attention_validation_scores.parquet


,history attention
model,history attention
history_events,50
embedding_dim,32
parameters,26777666
fit_searches,121989
smoke,False
device,cuda
minutes,0.6000
alpha,0.8715
best_epoch,1


,loss,inner_ndcg_at_10,seconds
epoch,,,
0,NaN,0.3366,0.0000
1,3.6125,0.3770,10.8000
2,3.5497,0.3756,10.8000
3,3.4930,0.3720,10.8000


In [11]:
per_search["history attention"] = evaluate(frame.assign(score=attention_scores), "score", expected_sizes=validation_sizes)
pairs = [("history attention", "feature MLP"), ("history attention", "lagged history popularity")]
if reference is not None:
    pairs += [("history attention", "LambdaMART M1"), ("history attention", "LambdaMART M1 + CF"),
              ("feature MLP", "LambdaMART M1 + CF")]
rows = {}
for a, b in pairs:
    d = paired_difference(per_search[a].loc[clean_ids], per_search[b].loc[clean_ids])
    rows[f"{a}  minus  {b}"] = {"A": per_search[a].ndcg.loc[clean_ids].mean(), "B": per_search[b].ndcg.loc[clean_ids].mean(),
                                "difference": d[0], "low": d[1], "high": d[2]}
active = clean_ids[(searches.loc[clean_ids].days_since_last_action < 7).to_numpy()]
for a, b in pairs[:1]:
    for label, ids in [("active in last 7 days", active), ("inactive 7+ days", clean_ids.difference(active))]:
        d = paired_difference(per_search[a].loc[ids], per_search[b].loc[ids])
        rows[f"{a}  minus  {b}, {label} ({len(ids):,})"] = {
            "A": per_search[a].ndcg.loc[ids].mean(), "B": per_search[b].ndcg.loc[ids].mean(),
            "difference": d[0], "low": d[1], "high": d[2]}
pd.DataFrame(rows).T

,A,B,difference,low,high
history attention minus feature MLP,0.3795,0.3889,-0.0095,-0.0113,-0.0075
history attention minus lagged history popularity,0.3795,0.3374,0.0420,0.0393,0.0448
history attention minus LambdaMART M1,0.3795,0.3887,-0.0092,-0.0112,-0.0073
history attention minus LambdaMART M1 + CF,0.3795,0.3918,-0.0123,-0.0143,-0.0103
feature MLP minus LambdaMART M1 + CF,0.3889,0.3918,-0.0028,-0.0044,-0.0012
"history attention minus feature MLP, active in last 7 days (16,534)",0.3831,0.3928,-0.0097,-0.0119,-0.0078
"history attention minus feature MLP, inactive 7+ days (2,310)",0.3538,0.3612,-0.0074,-0.0124,-0.0021


**Reading the result (E28, Colab T4).**
- The attention model reaches **0.380**, **-0.0095 [-0.0113, -0.0075]** below the feature MLP with the same inputs (active and inactive shoppers alike) and -0.012 below the final LambdaMART: the expectation (0 to +0.005 over the MLP) is not met.
- **It overfits from the first epoch:** the best inner NDCG@10 is after epoch 1 (0.377, already below the MLP's 0.382) and falls while the training loss keeps falling (3.61 to 3.49). Most of its 26.8M parameters are ID embeddings (530,542 products x 32 = 17M; shops 5.9M; brands 3.5M).
- **Design flaw:** candidate ID embeddings are trained on the labels of the training searches, all from the same day as the validation searches. They can act as a learned same-day engagement rate per product, shop or brand (like the non-deployable G2b, E15), which the deployable design excludes, and they memorise rather than generalise.
- One debugging cycle is allowed (PROJECT_PLAN 5c); proposed fix in the log (E29).

## 7. E29: history attention, debugged (the one debugging cycle)

E28 overfitted from the first epoch, and its candidate ID embeddings could act as a learned same-day engagement rate. Exactly two changes, everything else as in section 6:

1. **No product-ID embeddings.** An item is the sum of its shop, brand and L3-path embeddings (removes 17M of the 26.8M parameters). Similarity is learned at the level where pooled history matches helped (shop, brand, category).
2. **The candidate enters only through the history.** The scoring network gets the 33 transformed features, the attention summary u and the interaction c x u, but not the candidate representation c on its own. A shop or brand embedding can therefore no longer score candidates by itself.

In [12]:
FIELDS_V2 = ["shop", "brand", "L3"]  # columns 1-3 of the candidate and history id arrays (0 = product, not used)


class HistoryAttentionV2(HistoryAttention):
    """E29: items from shop, brand and L3 only; the candidate reaches the scoring network only via the history."""

    def __init__(self, n_inputs, dim=EMB_DIM, hidden=(128, 64), dropout=0.1):
        FeatureMLP.__init__(self, n_inputs + 2 * dim, hidden, dropout)
        self.items = nn.ModuleList([nn.Embedding(len(VOCAB[f]) + 1, dim, padding_idx=0) for f in FIELDS_V2])
        self.action_type = nn.Embedding(len(TYPE_ID) + 1, dim, padding_idx=0)
        self.age_bucket = nn.Embedding(len(AGE_EDGES) + 2, dim, padding_idx=0)
        self.term = nn.Embedding(len(VOCAB["term"]) + 1, dim, padding_idx=0)
        self.candidate_key, self.query_key = nn.Linear(dim, dim), nn.Linear(dim, dim)
        self.dim = dim

    def item(self, ids):
        return sum(embedding(ids[..., j + 1].long()) for j, embedding in enumerate(self.items))

    def forward(self, lists, index, mask, searches):
        c = self.item(lists.cand[index])
        hist = lists.hist[searches]
        events = (self.item(hist[..., :4]) + self.action_type(hist[..., 4].long())
                  + self.age_bucket(hist[..., 5].long()))
        has_event = hist[..., 4] > 0
        terms = lists.query[searches].long()
        q = self.term(terms).sum(dim=1) / (terms > 0).sum(dim=1, keepdim=True).clamp(min=1)
        key = self.candidate_key(c) + self.query_key(q)[:, None, :]
        logits = torch.einsum("bld,btd->blt", key, events) / self.dim ** 0.5
        logits = logits.masked_fill(~has_event[:, None, :], -1e9)
        weights = torch.softmax(logits, dim=-1) * has_event[:, None, :]
        u = torch.einsum("blt,btd->bld", weights, events)
        x = torch.cat([lists.x[index], u, c * u], dim=-1)  # no c on its own (E29)
        return self.alpha * lists.log_pop[index] + self.out(self.body(x)).squeeze(-1)


torch.manual_seed(MODEL_SEED)
check = predict(HistoryAttentionV2(data["fit"].x.shape[1]).to(DEVICE), data["validation"]).cpu().numpy()
assert np.allclose(check, np.log1p(np.nan_to_num(valid[2][:, pop_j], nan=0.0)), atol=1e-5)
print("untrained attention v2 = lagged popularity: ok")

saved = saved_run("attention_v2")
if saved is None:
    torch.manual_seed(MODEL_SEED)
    start = time.time()
    attention_v2, attention_v2_history = train(HistoryAttentionV2(data["fit"].x.shape[1]), label="attention v2")
    attention_v2_scores = predict(attention_v2, data["validation"]).cpu().numpy()
    attention_v2_summary = {"model": "history attention v2 (no product IDs)", "inputs": INPUTS,
                            "history_events": HIST_LEN, "embedding_dim": EMB_DIM,
                            "parameters": int(sum(p.numel() for p in attention_v2.parameters())),
                            "fit_searches": len(data["fit"]), "smoke": SMOKE, "device": str(DEVICE),
                            "minutes": round((time.time() - start) / 60, 1), "alpha": float(attention_v2.alpha.item()),
                            "best_epoch": int(attention_v2_history.inner_ndcg_at_10.idxmax())}
    save_run("attention_v2", attention_v2_scores, attention_v2_history, attention_v2_summary)
else:
    attention_v2_scores, attention_v2_history, attention_v2_summary = saved
display(pd.Series({k: v for k, v in attention_v2_summary.items() if k != "inputs"}).to_frame("history attention v2"))
attention_v2_history

untrained attention v2 = lagged popularity: ok
attention_v2: loaded from /content/drive/MyDrive/BT4222 Group Project/processed/neural/attention_v2_validation_scores.parquet


,history attention v2
model,history attention v2 (no product IDs)
history_events,50
embedding_dim,32
parameters,9796194
fit_searches,121989
smoke,False
device,cuda
minutes,0.6000
alpha,0.8662
best_epoch,2


,loss,inner_ndcg_at_10,seconds
epoch,,,
0,NaN,0.3366,0.0000
1,3.6085,0.3769,7.3000
2,3.5550,0.3791,7.0000
3,3.5271,0.3762,7.1000
4,3.4831,0.3718,7.2000


In [13]:
per_search["history attention v2"] = evaluate(frame.assign(score=attention_v2_scores), "score",
                                              expected_sizes=validation_sizes)
pairs = [("history attention v2", "feature MLP"), ("history attention v2", "history attention"),
         ("history attention v2", "lagged history popularity")]
if reference is not None:
    pairs += [("history attention v2", "LambdaMART M1"), ("history attention v2", "LambdaMART M1 + CF")]
rows = {}
for a, b in pairs:
    d = paired_difference(per_search[a].loc[clean_ids], per_search[b].loc[clean_ids])
    rows[f"{a}  minus  {b}"] = {"A": per_search[a].ndcg.loc[clean_ids].mean(), "B": per_search[b].ndcg.loc[clean_ids].mean(),
                                "difference": d[0], "low": d[1], "high": d[2]}
for label, ids in [("active in last 7 days", active), ("inactive 7+ days", clean_ids.difference(active))]:
    d = paired_difference(per_search["history attention v2"].loc[ids], per_search["feature MLP"].loc[ids])
    rows[f"history attention v2  minus  feature MLP, {label} ({len(ids):,})"] = {
        "A": per_search["history attention v2"].ndcg.loc[ids].mean(), "B": per_search["feature MLP"].ndcg.loc[ids].mean(),
        "difference": d[0], "low": d[1], "high": d[2]}
pd.DataFrame(rows).T

,A,B,difference,low,high
history attention v2 minus feature MLP,0.3816,0.3889,-0.0073,-0.0090,-0.0056
history attention v2 minus history attention,0.3816,0.3795,0.0021,0.0006,0.0038
history attention v2 minus lagged history popularity,0.3816,0.3374,0.0442,0.0413,0.0470
history attention v2 minus LambdaMART M1,0.3816,0.3887,-0.0071,-0.0090,-0.0050
history attention v2 minus LambdaMART M1 + CF,0.3816,0.3918,-0.0101,-0.0122,-0.0081
"history attention v2 minus feature MLP, active in last 7 days (16,534)",0.3855,0.3928,-0.0073,-0.0090,-0.0055
"history attention v2 minus feature MLP, inactive 7+ days (2,310)",0.3536,0.3612,-0.0076,-0.0127,-0.0030


**Reading the result (E29, Colab T4).**
- Without product IDs and without the candidate representation on its own, the attention model reaches **0.382**: +0.0021 [0.0006, 0.0038] over E28, but still **-0.0073 [-0.0090, -0.0056]** below the feature MLP with the same inputs (active and inactive shoppers alike) and -0.010 below the final LambdaMART. The expectation (about equal to the MLP) is not met.
- It still overfits, only later: the best inner NDCG@10 is after epoch 2 (0.379), then it falls while the training loss keeps falling (3.61 to 3.48). 9.8M parameters remain, mostly shop (5.9M) and brand (3.5M) embeddings.
- The debugging cycle is used up (PROJECT_PLAN 5c), so the neural part ends here. Reading: with one fixed, untuned configuration, learned weighting of individual history events does not beat four pooled counts and five CF similarities; the extra history block overfits rather than adds signal. This agrees with the trees (E18-E20) but does not rule out that a regularised attention model (embedding dropout, weight decay, smaller vocabularies) could match the MLP.

## 8. E31: regularised history attention (one pre-registered run)

E29 still overfits: the inner NDCG@10 peaks after epoch 2 and then falls while the training loss keeps falling, and most of its 9.8M parameters are shop and brand embeddings. E31 tests whether standard regularisation closes the gap to the feature MLP. The architecture is E29's (no product IDs, the candidate only through the history), with three changes, one configuration, no tuning, reported whatever the result:

1. **Smaller embeddings:** 16 instead of 32 dimensions (halves the embedding parameters).
2. **Embedding dropout 0.2** on the item representations of candidates and history events (training only).
3. **Weight decay on the embedding tables** (AdamW, decoupled, 0.1; the other weights without): an embedding row without a data signal shrinks by about 5% per epoch (476 steps x learning rate 0.001 x 0.1).

Everything else as in E29: inputs, loss, learning rate, batch size, early stopping, seed, populations and evaluation.

In [14]:
EMB_DIM_V3, EMBEDDING_DROPOUT, EMBEDDING_DECAY = 16, 0.2, 0.1


class HistoryAttentionV3(HistoryAttentionV2):
    """E31: E29 with 16-dimensional embeddings and dropout on the item representations (weight decay: see train)."""

    def __init__(self, n_inputs, dim=EMB_DIM_V3, hidden=(128, 64), dropout=0.1, embedding_dropout=EMBEDDING_DROPOUT):
        super().__init__(n_inputs, dim, hidden, dropout)
        self.embedding_dropout = nn.Dropout(embedding_dropout)

    def item(self, ids):
        return self.embedding_dropout(super().item(ids))


torch.manual_seed(MODEL_SEED)
check = predict(HistoryAttentionV3(data["fit"].x.shape[1]).to(DEVICE), data["validation"]).cpu().numpy()
assert np.allclose(check, np.log1p(np.nan_to_num(valid[2][:, pop_j], nan=0.0)), atol=1e-5)
print("untrained attention v3 = lagged popularity: ok")

saved = saved_run("attention_v3")
if saved is None:
    torch.manual_seed(MODEL_SEED)
    start = time.time()
    attention_v3, attention_v3_history = train(HistoryAttentionV3(data["fit"].x.shape[1]), label="attention v3",
                                               embedding_decay=EMBEDDING_DECAY)
    attention_v3_scores = predict(attention_v3, data["validation"]).cpu().numpy()
    attention_v3_summary = {"model": "history attention v3 (regularised)", "inputs": INPUTS,
                            "history_events": HIST_LEN, "embedding_dim": EMB_DIM_V3,
                            "embedding_dropout": EMBEDDING_DROPOUT, "embedding_decay": EMBEDDING_DECAY,
                            "parameters": int(sum(p.numel() for p in attention_v3.parameters())),
                            "fit_searches": len(data["fit"]), "smoke": SMOKE, "device": str(DEVICE),
                            "minutes": round((time.time() - start) / 60, 1), "alpha": float(attention_v3.alpha.item()),
                            "best_epoch": int(attention_v3_history.inner_ndcg_at_10.idxmax())}
    save_run("attention_v3", attention_v3_scores, attention_v3_history, attention_v3_summary)
else:
    attention_v3_scores, attention_v3_history, attention_v3_summary = saved
display(pd.Series({k: v for k, v in attention_v3_summary.items() if k != "inputs"}).to_frame("history attention v3"))
attention_v3_history

untrained attention v3 = lagged popularity: ok
attention v3 epoch 1: loss 3.6085, inner NDCG@10 0.3781, 6 s
attention v3 epoch 2: loss 3.5612, inner NDCG@10 0.3806, 6 s
attention v3 epoch 3: loss 3.5465, inner NDCG@10 0.3822, 6 s
attention v3 epoch 4: loss 3.5349, inner NDCG@10 0.3833, 6 s
attention v3 epoch 5: loss 3.5216, inner NDCG@10 0.3828, 6 s
attention v3 epoch 6: loss 3.5058, inner NDCG@10 0.3828, 6 s


,history attention v3
model,history attention v3 (regularised)
history_events,50
embedding_dim,16
embedding_dropout,0.2000
embedding_decay,0.1000
parameters,4903922
fit_searches,121989
smoke,False
device,cuda
minutes,0.7000


,loss,inner_ndcg_at_10,seconds
epoch,,,
0,NaN,0.3366,0.0000
1,3.6085,0.3781,6.0000
2,3.5612,0.3806,5.7000
3,3.5465,0.3822,5.8000
4,3.5349,0.3833,5.8000
5,3.5216,0.3828,5.8000
6,3.5058,0.3828,5.8000


In [15]:
per_search["history attention v3"] = evaluate(frame.assign(score=attention_v3_scores), "score",
                                              expected_sizes=validation_sizes)
pairs = [("history attention v3", "feature MLP"), ("history attention v3", "history attention v2"),
         ("history attention v3", "lagged history popularity")]
if reference is not None:
    pairs += [("history attention v3", "LambdaMART M1"), ("history attention v3", "LambdaMART M1 + CF")]
rows = {}
for a, b in pairs:
    d = paired_difference(per_search[a].loc[clean_ids], per_search[b].loc[clean_ids])
    rows[f"{a}  minus  {b}"] = {"A": per_search[a].ndcg.loc[clean_ids].mean(), "B": per_search[b].ndcg.loc[clean_ids].mean(),
                                "difference": d[0], "low": d[1], "high": d[2]}
for label, ids in [("active in last 7 days", active), ("inactive 7+ days", clean_ids.difference(active))]:
    d = paired_difference(per_search["history attention v3"].loc[ids], per_search["feature MLP"].loc[ids])
    rows[f"history attention v3  minus  feature MLP, {label} ({len(ids):,})"] = {
        "A": per_search["history attention v3"].ndcg.loc[ids].mean(), "B": per_search["feature MLP"].ndcg.loc[ids].mean(),
        "difference": d[0], "low": d[1], "high": d[2]}
pd.DataFrame(rows).T

,A,B,difference,low,high
history attention v3 minus feature MLP,0.3867,0.3889,-0.0023,-0.0035,-0.0010
history attention v3 minus history attention v2,0.3867,0.3816,0.0051,0.0033,0.0067
history attention v3 minus lagged history popularity,0.3867,0.3374,0.0492,0.0461,0.0524
history attention v3 minus LambdaMART M1,0.3867,0.3887,-0.0020,-0.0037,-0.0003
history attention v3 minus LambdaMART M1 + CF,0.3867,0.3918,-0.0051,-0.0068,-0.0033
"history attention v3 minus feature MLP, active in last 7 days (16,534)",0.3904,0.3928,-0.0024,-0.0037,-0.0011
"history attention v3 minus feature MLP, inactive 7+ days (2,310)",0.3597,0.3612,-0.0015,-0.0051,0.0022


**Reading the result (E31, Colab T4).**
- Regularised, the attention model reaches **0.387**: **+0.0051 [0.0033, 0.0067]** over E29, but still **-0.0023 [-0.0035, -0.0010]** below the feature MLP (active -0.0024, inactive -0.0015 with an interval including 0) and -0.0051 below the final LambdaMART. Within the expectation (0.383-0.389, -0.006 to 0 vs the MLP); attention does not add (rule: at least +0.003 over the MLP).
- **The overfitting is largely gone:** the inner NDCG@10 rises to epoch 4 (0.383, E29: 0.379 at epoch 2) and then stays flat (0.383, 0.383) instead of falling, while the training loss falls more slowly (3.61 to 3.51 over six epochs; E29 3.61 to 3.48 over four). Half the parameters (4.9M).
- Reading: the overfitting diagnosis is confirmed, and regularisation recovers about two thirds of the gap to the MLP (-0.0073 to -0.0023). Even regularised, learned weighting of individual events stays slightly below pooled counts and CF; the extra history block adds no signal the inputs do not already carry. The neural part ends here (pre-registered).